# Session 9: Experimental design and statistical inference

A state regulator decides whether a city's water exceeds the lead action level from the 90th percentile of the lead measured in tap water at a sample of homes, and a different sample of homes would give a different 90th percentile. This notebook computes that percentile by the rule's rank method, puts a lower bound on it, and asks whether the gap between a first-draw and a flushed sample from the same homes is one the call depends on. Section 1 works through all 271 samples Virginia Tech collected in Flint, Michigan, in 2015. Section 2 does the same for 60 of those samples assigned to you, a sampling campaign a utility might run. Section 3 compares a paired and a two-sample t-test of first-draw against flushed lead on your homes. Section 4 is the judgment cell.

The lead results are real, measured by ICP-MS: FlintWaterStudy.org (2015), "Lead Results from Tap Water Sampling in Flint, MI during the Flint Water Crisis", Virginia Tech, file `Flint-Samples-FINAL.xlsx` ([download link on FlintWaterStudy.org](https://flintwaterstudy.org/wp-content/uploads/2015/12/Flint-Samples-FINAL.xlsx)), no licence stated, accessed 2026-10-10. Because the site states no licence, the file is not in the repository. The first code cell of section 1 downloads it into `data/` and checks it. `data/draws.csv` lists the rows of each student's draw; `data/README.md` describes both.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 9 assignment on Canvas by 1:30 pm on Tuesday, November 3. Submissions are individual. Every draw's rows are in the repository, and the graders' run downloads the same file and checks it byte for byte, so it reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas and Matplotlib, and three names from SciPy's statistics module that are new this session. `binom` is the binomial distribution, which gives the rank of a lower bound on a percentile. `ttest_rel` and `ttest_ind` are the paired and the two-sample t-tests; each returns a result whose `.pvalue` is the test's p value and whose `.confidence_interval()` is the 95% interval on the difference in means. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, ttest_ind, ttest_rel

## Parameters

The fetched file and the table of draws; the rule's percentile and the two action levels, fixed here before any data are read; the confidence of the lower bound; the detection limit Virginia Tech stated; then your draw, set in section 2. Leave `DRAW` as `CHANGE-ME` until section 2.

In [ ]:
FLINT_XLSX = Path("data/Flint-Samples-FINAL.xlsx")   # fetched by data/fetch_flint.py in section 1, ignored by git
DRAWS_CSV = Path("data/draws.csv")             # each draw's 60 samples, by their row in the spreadsheet
PERCENTILE = 0.9                               # 40 CFR 141.80(c)(3): the 90th percentile of the samples
ACTION_LEVEL_2015_UG_L = 15.0                  # the lead action level in 2015, ug/L
ACTION_LEVEL_LCRI_UG_L = 10.0                  # the action level from 2027-11-01 (LCRI), ug/L
CONFIDENCE = 0.95                              # of the one-sided lower bound on the 90th percentile
STATED_LIMIT_UG_L = 1.0                        # Virginia Tech's stated detection limit for lead, ug/L
DRAW = "CHANGE-ME"                             # your assigned draw, for example "draw_07", section 2

## 1. Explore: read the 90th percentile of a city's samples

Virginia Tech's volunteers took three bottles at each home in August and September 2015: the first draw after at least 6 hours of stagnation, then the water after 45 s and after 2 min of flushing. The first cell runs `data/fetch_flint.py`, which downloads the spreadsheet once and checks that it is the file the course recorded. It then reads sheet 1, one row per sample, with `load_homes`, a helper given complete, which keeps each sample's row number in the spreadsheet so that section 2 can pick out your draw. Sheet 2 holds 5 samples the study did not use.

In [ ]:
%run data/fetch_flint.py


def load_homes(path):
    """One row per sample in sheet 1 of the Flint file, with its row number in the sheet.

    Parameters: path, the fetched Flint-Samples-FINAL.xlsx.
    Returns: a DataFrame with sheet_row, sample_id, and lead in the three bottles in ug/L.
    """
    sheet = pd.read_excel(path, sheet_name=0)
    homes = sheet[pd.to_numeric(sheet["SampleID"], errors="coerce").notna()].copy()   # the rows below the samples are credits
    homes["sheet_row"] = homes.index + 2   # row 1 of the sheet is the header
    homes = homes.rename(columns={
        "SampleID": "sample_id",
        "Pb Bottle 1 (ppb) - First Draw": "first_draw_ug_L",
        "Pb Bottle 2 (ppb) - 45 secs flushing": "flushed_45s_ug_L",
        "Pb Bottle 3 (ppb) - 2 mins flushing": "flushed_2min_ug_L",
    })
    bottles = ["first_draw_ug_L", "flushed_45s_ug_L", "flushed_2min_ug_L"]
    homes[bottles] = homes[bottles].astype(float)
    return homes[["sheet_row", "sample_id"] + bottles].reset_index(drop=True)


homes = load_homes(FLINT_XLSX)
not_used = pd.read_excel(FLINT_XLSX, sheet_name=1)
print(f"sheet 1: {len(homes)} samples from {homes['sample_id'].nunique()} homes; sheet 2: {len(not_used)} samples not used")
print(homes.head().to_string(index=False))

The rule does not average the samples. By 40 CFR 141.80(c)(3), the results are sorted from lowest to highest, the number of samples is multiplied by 0.9, and the value at that rank is the 90th percentile. With 271 samples the rank is 0.9 × 271 = 243.9, which is not a whole number, and the rule does not say how to read it. The next cell prints the mean and the median, then four readings of rank 243.9 and NumPy's `np.percentile`, and plots every first draw against its rank on a log axis, with both action levels.

In [ ]:
first_sorted = np.sort(homes["first_draw_ug_L"].to_numpy())   # ug/L, lowest first; rank 1 is first_sorted[0]
n_all = len(first_sorted)
rank_all = PERCENTILE * n_all                                     # 243.9 for 271 samples
lower_rank, upper_rank = int(rank_all), int(rank_all) + 1         # the whole ranks on either side
at_lower, at_upper = first_sorted[lower_rank - 1], first_sorted[upper_rank - 1]
minitab_rank = PERCENTILE * (n_all + 1)                          # another common convention, 0.9 (n + 1)
at_minitab = first_sorted[int(minitab_rank) - 1] + (minitab_rank % 1) * (first_sorted[int(minitab_rank)] - first_sorted[int(minitab_rank) - 1])
readings = {
    f"truncate to rank {lower_rank}": at_lower,
    f"interpolate between ranks {lower_rank} and {upper_rank}": at_lower + (rank_all - lower_rank) * (at_upper - at_lower),
    f"round up to rank {upper_rank}": at_upper,
    f"0.9 (n + 1), rank {minitab_rank:.1f}": at_minitab,
}
print(f"mean {first_sorted.mean():.2f} ug/L, median {np.median(first_sorted):.2f} ug/L")
for level in [ACTION_LEVEL_2015_UG_L, ACTION_LEVEL_LCRI_UG_L]:
    print(f"{(first_sorted > level).sum()} of {n_all} samples above {level:g} ug/L")
print(f"0.9 x {n_all} = {rank_all:.1f}; rank {lower_rank} holds {at_lower} and rank {upper_rank} holds {at_upper} ug/L")
for name, value in readings.items():
    print(f"  90th percentile, {name}: {value:.2f} ug/L")
print(f"  np.percentile(first_sorted, 90): {np.percentile(first_sorted, 100 * PERCENTILE):.2f} ug/L")

fig, ax = plt.subplots()
ax.plot(np.arange(1, n_all + 1), first_sorted, "o", color="black", fillstyle="none", markersize=4, label="first draw, one sample")
ax.axvline(rank_all, color="gray", linestyle=":", label=f"rank {rank_all:.1f}")
ax.axhline(ACTION_LEVEL_2015_UG_L, color="black", linestyle="--", linewidth=1, label="15 µg/L, 2015")
ax.axhline(ACTION_LEVEL_LCRI_UG_L, color="black", linestyle="-.", linewidth=1, label="10 µg/L, from 2027")
ax.set_yscale("log")
ax.set_xlabel("rank, lowest first")
ax.set_ylabel("first-draw lead (µg/L)")
ax.set_title("Flint, 2015: 271 first-draw samples by rank")
ax.legend()
fig.tight_layout()
plt.show()

The 271 first draws rise from 0.344 µg/L at rank 1 to 158 µg/L at rank 271 on the log axis, crossing 10 µg/L at rank 208 and 15 µg/L at rank 227, so 45 samples sit above 15 µg/L and rank 243.9, the 90th percentile's rank, falls among values of 25 to 27 µg/L. The four readings of that rank run from 25.21 to 26.94 µg/L and `np.percentile` gives 26.64 µg/L, the same as rounding up at this n, and every one of them is above both action levels.

**Explain.** In two sentences: the mean and the 90th percentile of the same 271 samples fall on opposite sides of 15 µg/L. Why, from the plot, and which one answers the question the rule asks?

*Your answer:*

Virginia Tech stated a detection limit of 1 µg/L but reported every result as a number, including those below it. Session 5's rule, that a result below the reporting limit is not zero, is for a result written as `<` a limit; here nothing is censored, so the values are kept as reported. To check that the choice could not change the call, the next cell sets the results below 1 µg/L to 0, to half the limit and to the limit, and prints the 90th percentile at rank 244 and the mean of first draw minus 2 min flushed under each.

In [ ]:
n_below = int((homes["first_draw_ug_L"] < STATED_LIMIT_UG_L).sum())
print(f"{n_below} of {n_all} first draws below {STATED_LIMIT_UG_L:g} ug/L")
for label, fill in [("as reported", None), ("set to 0", 0.0), ("set to half the limit", 0.5 * STATED_LIMIT_UG_L), ("set to the limit", STATED_LIMIT_UG_L)]:
    first = homes["first_draw_ug_L"].to_numpy()
    flushed = homes["flushed_2min_ug_L"].to_numpy()
    if fill is not None:
        first = np.where(first < STATED_LIMIT_UG_L, fill, first)
        flushed = np.where(flushed < STATED_LIMIT_UG_L, fill, flushed)
    print(f"below the limit {label:>22}: 90th percentile {np.sort(first)[upper_rank - 1]:.2f} ug/L, "
          f"mean first draw minus 2 min {np.mean(first - flushed):.2f} ug/L")

The rule takes the 90th percentile as it is: above the level is an exceedance. A different 271 homes would have given a different one, and a lower bound says how strongly these samples show that the 90th percentile they were drawn from is above the level. If the samples are a random draw, the number below that true 90th percentile is binomial, with n trials and probability 0.9. `binom.ppf(0.05, n, 0.9)` is the largest rank r with at least r samples below it with 95% probability, so the value at rank r is a one-sided 95% lower bound.

In [ ]:
rank_bound_all = int(binom.ppf(1 - CONFIDENCE, n_all, PERCENTILE))   # 236 for 271 samples
bound_all = first_sorted[rank_bound_all - 1]
coverage = 1 - binom.cdf(rank_bound_all - 1, n_all, PERCENTILE)       # probability that at least that many fall below
print(f"rank {rank_bound_all} of {n_all}: one-sided {CONFIDENCE:.0%} lower bound on the 90th percentile, {bound_all:.2f} ug/L")
print(f"probability that at least {rank_bound_all} samples fall below the true 90th percentile: {coverage:.3f}")
for level in [ACTION_LEVEL_2015_UG_L, ACTION_LEVEL_LCRI_UG_L]:
    print(f"lower bound above {level:g} ug/L: {bound_all > level}")

## 2. Apply: call the city from your own 60 samples

Your draw is 60 of the same 271 samples, chosen at random without replacement, the size of a campaign a utility might run. The city is the same for everyone, but each draw gives its own 90th percentile and its own lower bound, and some draws show an exceedance of 15 µg/L at 95% that others do not. With 60 samples, 0.9 × 60 = 54 is a whole number, so the rule's rank has one reading.

**[TASK 1]** Set `DRAW` in the Parameters cell to your assigned draw, run the Parameters cell, then run the next cell. Until `DRAW` is set, it selects no rows and stops with `AssertionError: no draw named 'CHANGE-ME' in data/draws.csv; set DRAW in Parameters`.

In [ ]:
draws = pd.read_csv(DRAWS_CSV)
my_rows = draws.loc[draws["draw"] == DRAW, "sheet_row"]
assert len(my_rows) > 0, f"no draw named {DRAW!r} in {DRAWS_CSV.as_posix()}; set DRAW in Parameters"
my_homes = homes[homes["sheet_row"].isin(my_rows)].reset_index(drop=True)
n = len(my_homes)
print(f"{DRAW}: {n} samples, first-draw mean {my_homes['first_draw_ug_L'].mean():.2f} ug/L, "
      f"highest {my_homes['first_draw_ug_L'].max():.2f} ug/L")

**[TASK 2]** Read your draw's 90th percentile by the rule's rank. The cell sorts your first draws and computes the rank; reading the value at that rank is left to you. Until it is filled, the cell stops with `TypeError: float() argument must be a string or a real number, not 'ellipsis'`.

In [ ]:
my_sorted = np.sort(my_homes["first_draw_ug_L"].to_numpy())   # ug/L, lowest first; rank 1 is my_sorted[0]
rank_p90 = int(PERCENTILE * n)                                   # 0.9 x 60 = 54
p90_ug_L = float(...)   # [TASK 2] the value at rank rank_p90 in my_sorted, by the rule in notes section 2
print(f"rank {rank_p90} of {n}: 90th percentile {p90_ug_L:.2f} ug/L")

**Ask the assistant** now that TASK 2 has given you the 90th percentile, in Claude Code, and give the assistant a check to run, today's AI practice, met in session 6. First save the notebook (**File**, then **Save**), so the assistant reads the `DRAW` you set. Then ask:

`In my notebook, my_homes holds my draw of homes, with first-draw lead in first_draw_ug_L. Compute the 90th percentile of first-draw lead by the Lead and Copper Rule's rank method, then add a cell to my notebook that prints the sorted values at ranks 52 to 57 with their rank numbers, so I can check it.`

The assistant adds a cell at the end of the notebook and asks you to click **Execute** or **Cancel**. If no **Execute** button appears, or the assistant shows numbers from a script it ran elsewhere, run its cell yourself, or click **+ Code** in the notebook toolbar, paste its code and run that. Read rank 54 in the output under the cell, not in the chat: the printout is the check, and the chat's number is a claim. Compare three numbers: the printout's rank 54, your `p90_ug_L` from TASK 2, and the number the assistant reported. Then delete the added cell (the trash-can icon, **Delete Cell**, in the cell's toolbar), so that the notebook still ends with the software environment cell. What it suggested and what you did with it is one candidate for part (b) of the judgment cell.

**[TASK 3]** Put the one-sided 95% lower bound on your 90th percentile, as section 1 did for all 271 samples. The cell computes the bound's rank from `binom`; reading the value at that rank is left to you. Until it is filled, the cell stops with `TypeError: float() argument must be a string or a real number, not 'ellipsis'`.

In [ ]:
rank_bound = int(binom.ppf(1 - CONFIDENCE, n, PERCENTILE))      # 50 for 60 samples
bound_ug_L = float(...)   # [TASK 3] the value at rank rank_bound in my_sorted
print(f"rank {rank_bound} of {n}: one-sided {CONFIDENCE:.0%} lower bound {bound_ug_L:.2f} ug/L")

The next cell draws your 60 first draws by rank, as section 1 drew all 271, with the rule's rank, the bound's rank and both action levels.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, n + 1), my_sorted, "o", color="black", fillstyle="none", label="first draw, one sample")
ax.axvline(rank_p90, color="gray", linestyle=":", label=f"rank {rank_p90}, the 90th percentile")
ax.axvline(rank_bound, color="gray", linestyle="-", linewidth=1, label=f"rank {rank_bound}, the lower bound")
ax.axhline(ACTION_LEVEL_2015_UG_L, color="black", linestyle="--", linewidth=1, label="15 µg/L, 2015")
ax.axhline(ACTION_LEVEL_LCRI_UG_L, color="black", linestyle="-.", linewidth=1, label="10 µg/L, from 2027")
ax.set_yscale("log")
ax.set_xlabel("rank, lowest first")
ax.set_ylabel("first-draw lead (µg/L)")
ax.set_title(f"{DRAW}: 60 first-draw samples by rank")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**[TASK 4]** Make the call against each action level, twice. The rule's call is given: the cell prints whether your 90th percentile is above the level, which under 141.80(c) is an exceedance. Whether your lower bound shows the exceedance at 95% is left to you. Until it is filled, the cell stops at the line after it with `KeyError: Ellipsis`.

In [ ]:
for level in [ACTION_LEVEL_2015_UG_L, ACTION_LEVEL_LCRI_UG_L]:
    supported = ...   # [TASK 4] does your lower bound show the 90th percentile above level? True or False
    shown = {True: "shown", False: "not shown"}[supported]
    rule = "an exceedance" if p90_ug_L > level else "no exceedance"
    print(f"{level:g} ug/L: 90th percentile {p90_ug_L:.2f}, {rule} under the rule; "
          f"lower bound {bound_ug_L:.2f}, exceedance {shown} at {CONFIDENCE:.0%}")

**Explain.** Before you compare: in two sentences, give your 90th percentile and your lower bound, and say for 15 µg/L and for 10 µg/L whether your draw is an exceedance under the rule and whether the bound shows it at 95%. Then compare with a neighbor, if you can one whose bound falls on the other side of 15 µg/L: set your draws, 90th percentiles and lower bounds side by side. Add one sentence on where your two answers differ and what in your two draws explains it.

*Your answer:*

## 3. Analyze: choose a paired or a two-sample test for flushing

Every home in your draw gave a first draw and a 2 min flushed sample, and the rule counts only the first. To ask how much lead the stagnant water adds, the test has to follow how the samples were taken. A two-sample t-test, the one two columns in a spreadsheet invite, treats the first draws and the flushed samples as two separate groups of 60. A paired t-test takes each home's first draw minus its own flushed sample and tests the 60 differences, which removes the home-to-home spread. The next cell runs both, so you see what ignoring the pairing costs, then prints the 90th percentile of each bottle at rank 54 and plots each home's two samples against each other.

In [ ]:
first = my_homes["first_draw_ug_L"]
flushed = my_homes["flushed_2min_ug_L"]
widths = {}
for name, result in [("paired", ttest_rel(first, flushed)), ("two-sample", ttest_ind(first, flushed))]:
    interval = result.confidence_interval()                           # 95%, on first draw minus flushed, ug/L
    widths[name] = interval.high - interval.low
    print(f"{name:>10}: mean difference {(first - flushed).mean():.2f} ug/L, 95% interval {interval.low:.2f} to "
          f"{interval.high:.2f} ug/L (width {widths[name]:.2f}), p = {result.pvalue:.2g}")
print(f"paired width over two-sample width: {widths['paired'] / widths['two-sample']:.2f}")
for column in ["first_draw_ug_L", "flushed_45s_ug_L", "flushed_2min_ug_L"]:
    print(f"{column:>18}: 90th percentile at rank {rank_p90}, {np.sort(my_homes[column].to_numpy())[rank_p90 - 1]:.2f} ug/L")

fig, ax = plt.subplots()
ax.plot(first, flushed, "o", color="black", fillstyle="none", label="one home")
span = [min(first.min(), flushed.min()), max(first.max(), flushed.max())]
ax.plot(span, span, "-", color="gray", label="flushed equals first draw")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("first-draw lead (µg/L)")
ax.set_ylabel("lead after 2 min flushing (µg/L)")
ax.set_title(f"{DRAW}: each home's first draw and 2 min flushed sample")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** In two sentences, say which test fits these data and why, citing the two intervals' widths and p values, and whether the first draw against flushed difference is large enough in µg/L to change the call, citing your 2 min flushed 90th percentile beside your first-draw one and the two action levels. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the regulator, is your draw of 60 samples an exceedance of 15 µg/L and of 10 µg/L under the rule, and does its lower bound show the exceedance at 95%? Cite your 90th percentile at rank 54 and its one-sided 95% lower bound, and the mean difference between first-draw and 2 min flushed lead with the interval from the test you chose, and say what the same rule would have called on the 2 min flushed bottle. Then say what your call means for the city's residents, and what a lower bound on the other side of 15 µg/L would have changed. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. The Ask the assistant cell after TASK 2 is the natural source; the reason has to cite a value from your output, such as the printed value at rank 54 against the number the assistant reported, not a general rule. This session's AI practice, named in the notes, is *give the assistant a check to run*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")